# Flagship loss model notebook

REFERENCE ONLY: simulate and review before use.

This notebook drives the same code the tests check: the analytic model, the time-domain circuit simulation, and the efficiency curve. Run it from the repo root after `pip install -e .[dev]`.

In [ ]:
from watt_forge.flagship import model, sim, Params
from watt_forge.style import group_losses
p = Params()
r = model.best(56.0, 48.0, 400.0, p)
print(r['label'], r['fs'], round(100*r['eta'], 3))
for k, v in sorted(r['losses'].items(), key=lambda kv: -kv[1]):
    print(f'{k:20s} {v:8.3f} W')

## Circuit simulation vs analytic model
The simulator solves the periodic steady state of the real switched circuit (dead time, reverse conduction, unbalanced flying capacitors) and must agree with the model to within 0.15 points.

In [ ]:
for vin, frac in [(12, 1), (24, 1), (36, .5), (48, 1), (56, 1), (60, .1)]:
    pout = frac * p.spec.p_rated(vin)
    a = model.best(vin, 48.0, pout, p)
    s = sim.simulate(vin, 48.0, pout, a['fs'], a['mode'], p, a['phase'])
    print(f"{vin:5.1f} V {a['label']:24s} model {100*a['eta']:.3f} %  sim {100*s['eta']:.3f} %")

## Efficiency curve vs input voltage

In [ ]:
import matplotlib.pyplot as plt
vins = list(range(12, 61, 2))
for frac in (0.1, 0.3, 0.5, 1.0):
    plt.plot(vins, [100*model.best(v, 48.0, frac*p.spec.p_rated(v), p)['eta'] for v in vins], label=f'{int(frac*100)} %')
plt.xlabel('Vin [V]'); plt.ylabel('efficiency [%]'); plt.legend(); plt.grid(True); plt.show()

## Change a parameter and watch the budget move
Try `p.switch = 'EPC2302'`, `p.t_dead = 20e-9` or `p.l = 2.2e-6` and re-run.

In [ ]:
from dataclasses import replace
for dev in ['EPC2361', 'EPC2302', 'EPC2088', 'ISC030N10NM6']:
    q = replace(p, switch=dev)
    r = model.best(36.0, 48.0, 300.0, q)
    print(dev, round(100*r['eta'], 3), {k: round(v, 3) for k, v in group_losses(r['losses'])})